In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1. Đọc Bronze dạng Stream
df_stream = spark.readStream.format("delta").table("bronze_crypto_prices")

# 2. Xử lý logic
df_silver = df_stream \
                    .withWatermark("timestamp", "1 minutes") \
                    .withColumn("category",
                                F.when(F.col("symbol").isin(['BTCUSDT', 'ETHUSDT']), "Majorcoin")
                                .when(F.col("symbol").isin(['BNBUSDT', 'SOLUSDT', 'ADAUSDT']), "Altcoin")
                                .when(F.col("symbol").isin(['DOGEUSDT', 'PEPEUSDT']), "Memecoin")
                                .otherwise("Stablecoin")) \
                    .withColumn("processed_at", F.current_timestamp()) \
                    .dropDuplicates(["symbol", "timestamp"])

# 3. Ghi dữ liệu
checkpoint_path = "Files/checkpoint/crypto_silver_v1"

query = df_silver.writeStream \
                .format("delta") \
                .outputMode("append") \
                .option("checkpointLocation", checkpoint_path) \
                .option("mergeSchema", "true") \
                .trigger(processingTime='1 minute') \
                .toTable("silver_crypto_prices")

StatementMeta(, 8b10cc86-2b8c-49a3-b094-7d596505f117, 3, Finished, Available, Finished, False)